# Pair IE vs sum of single IEs (exact patching)

**EXPLORATORY** (CLAUDE.md §2: exempt from the TDD gate).

Reads `scripts/exact_ie.py` output: `outputs/exact_ie/size1` (single-node IEs) and
`outputs/exact_ie/size2` (all pairs of moving nodes). For each pair $\{i,j\}$:
x = $\mathrm{IE}(i)+\mathrm{IE}(j)$, y = $\mathrm{IE}(\{i,j\})$, diff = x − y.

- Rows where **both** |x| and |y| are below `THRESH` are dropped. Filtering on *either* would
  drop exactly the redundancy cases (zero singles, nonzero pair).
- Pair IE is stored **float16** (~3 significant digits), so |diff| below ≈ |y|·5e-4 is rounding noise.
- `reachable` pairs are do-interventions where clamping the downstream node blocks the upstream
  node's path, so they are not expected to be additive anyway. The scatter colours them separately.
- Two R² values: against y = x (does additivity hold?) and the squared Pearson r (best linear fit).

In [ ]:
import json
import os
import sys
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from transformers import AutoTokenizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)                      # config data paths are repo-relative
sys.path.insert(0, str(ROOT))
from src.data.loaders import load_examples

S1, S2 = Path("outputs/exact_ie/size1"), Path("outputs/exact_ie/size2")
THRESH = 1e-6        # drop a pair if |sum| < THRESH and |set IE| < THRESH
TOP = 25             # rows shown from each end of the diff ordering
MAX_PLOT = 200_000   # points drawn per scatter (R² always uses every row)
SYMLOG = False       # symlog axes (linthresh = THRESH) for IEs spanning many decades
KEY = ["site_ordinal", "token_idx", "feature_idx"]

## Match examples and recover their prompts

In [ ]:
def replay_examples(root):
    # Re-run the loader exactly as exact_ie.py did (same data config + seed). The
    # index of an example is only meaningful under its own run's config, because
    # use_min_length_only + num_examples changes which examples survive.
    meta = json.loads((root / "metadata.json").read_text(encoding="utf-8"))
    cfg = meta["config"]
    assert cfg["suite"]["name"] == "pythia-marks", cfg["suite"]["name"]
    # The loader only uses .tokenizer; lengths are compared relatively, so a BOS
    # difference from TransformerLens' tokenizer cannot change which examples survive.
    tok = SimpleNamespace(tokenizer=AutoTokenizer.from_pretrained("EleutherAI/pythia-70m-deduped"))
    return load_examples(SimpleNamespace(**cfg["data"]), tok, cfg["seed"])

def done(root, fname):
    return {p.parent.name for p in root.glob(f"ex*/{fname}")}

d1, d2 = done(S1, "singles.parquet"), done(S2, "pairs.parquet")
shared = sorted(d1 & d2)
print(f"{len(shared)} examples in both; only size1: {len(d1 - d2)}, only size2: {len(d2 - d1)}")

ex1, ex2 = replay_examples(S1), replay_examples(S2)
examples = {}
for name in shared:
    k = int(name[2:])
    a, b = ex1[k], ex2[k]
    if (a.clean_prefix, a.patch_prefix) != (b.clean_prefix, b.patch_prefix):
        raise ValueError(f"{name}: size1 and size2 runs loaded different prompts")
    examples[name] = a

## Load pairs

In [ ]:
def load_pairs(name):
    n1 = pd.read_parquet(S1 / name / "nodes.parquet")
    s1 = pd.read_parquet(S1 / name / "singles.parquet").sort_values("node")
    assert (s1.node.to_numpy() == np.arange(len(n1))).all()
    n1["ie"] = s1.ie.to_numpy()
    n2 = pd.read_parquet(S2 / name / "nodes.parquet")
    # The two runs computed moving_nodes separately: match nodes by identity, not row number.
    ie1 = n2[KEY].merge(n1[KEY + ["ie"]], on=KEY, how="left")["ie"].to_numpy()
    assert len(ie1) == len(n2)

    p = pd.read_parquet(S2 / name / "pairs.parquet")
    i, j = p.i.to_numpy(), p.j.to_numpy()
    df = pd.DataFrame({"i": i, "j": j, "sum_ie": ie1[i] + ie1[j],
                       "set_ie": p.ie.to_numpy().astype(np.float32), "reachable": p.reachable})
    n_pairs, n_missing = len(df), int(df.sum_ie.isna().sum())
    df = df[df.sum_ie.notna() & ((df.sum_ie.abs() >= THRESH) | (df.set_ie.abs() >= THRESH))]
    df = df.assign(diff=df.sum_ie - df.set_ie)
    print(f"{name}: {n_pairs:,} pairs, {n_missing:,} with a node missing from size1, "
          f"{len(df):,} kept")
    return df, n2

# ponytail: every example's filtered pairs held in RAM; stream per example if it gets tight.
pairs, nodes = {}, {}
for name in shared:
    pairs[name], nodes[name] = load_pairs(name)

## Helpers

In [ ]:
def r2(x, y):
    x, y = np.asarray(x, np.float64), np.asarray(y, np.float64)
    r2_identity = 1 - ((y - x) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return r2_identity, np.corrcoef(x, y)[0, 1] ** 2

def scatter(df, title):
    r2_id, r2_fit = r2(df.sum_ie, df.set_ie)
    s = df.sample(MAX_PLOT, random_state=0) if len(df) > MAX_PLOT else df
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    for reach, g in s.groupby("reachable"):
        ax.scatter(g.sum_ie, g.set_ie, s=3, alpha=0.4, rasterized=True,
                   label=f"{'reachable' if reach else 'antichain'} ({(df.reachable == reach).sum():,})")
    ax.axline((0, 0), slope=1, color="k", lw=0.8, ls="--", label="y = x")
    if SYMLOG:
        ax.set_xscale("symlog", linthresh=THRESH)
        ax.set_yscale("symlog", linthresh=THRESH)
    ax.set(xlabel="IE(i) + IE(j)", ylabel="IE({i, j})",
           title=f"{title}\nn = {len(df):,}   R²(y=x) = {r2_id:.4f}   R²(linear fit) = {r2_fit:.4f}")
    ax.legend(markerscale=4)
    plt.show()

def label(rows, name_col=None, name=None):
    # Attach (site, token, feature) of i and j to the few rows being shown.
    out = rows.copy()
    for c in ("i", "j"):
        info = [nodes[n].iloc[k] for n, k in zip(out[name_col] if name_col else [name] * len(out), out[c])]
        out[f"{c}_node"] = [f"s{int(r.site_ordinal)}/t{int(r.token_idx)}/"
                            f"{'err' if r.feature_idx < 0 else int(r.feature_idx)}" for r in info]
    return out

def tables(df, **kw):
    cols = (["example"] if "example" in df else []) + ["i_node", "j_node", "sum_ie", "set_ie", "diff", "reachable"]
    display(Markdown("**Largest diff (sum − set), descending**"))
    display(label(df.nlargest(TOP, "diff"), **kw)[cols])
    display(Markdown("**Smallest diff (bottom of the same ordering)**"))
    display(label(df.nsmallest(TOP, "diff").iloc[::-1], **kw)[cols])

## All prompts pooled

In [ ]:
pooled = pd.concat(pairs, names=["example"]).reset_index(level=0)
scatter(pooled, f"All prompts ({len(shared)})")
tables(pooled, name_col="example")

## Per prompt

In [ ]:
for name in shared:
    e = examples[name]
    display(Markdown(f"### {name}\n"
                     f"- clean: `{e.clean_prefix}` → `{e.clean_answer}`\n"
                     f"- counterfactual: `{e.patch_prefix}` → `{e.patch_answer}`"))
    scatter(pairs[name], name)
    tables(pairs[name], name=name)